# Evaluating frontier LLMs

In [ ]:
import os
import time
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
from datetime import datetime
import json
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

data_path = "../data"

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

In [6]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL)

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


### Transform data into appropriate format

In [2]:
def prompt_transform(target_set, test):

    modified_set = []

    for item in target_set:
        headlines = "\n\n".join([f"{i + 1}: {headline}" for i, headline in enumerate(item["headlines"])])
        headlines = f"Identify which of the four following headlines is fake:\n\n{headlines}\n\nAnswer: "
        if test:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
            }
        else:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
                "completion": str(item["answer"] + 1)
            }

        modified_set.append(entry)

    return modified_set

test_set_frontier_LLM_ready = prompt_transform(test_set, True)

Create structured output

In [9]:
class FakeHeadlineOutput(BaseModel):
    """Structured numeric answer for predicting which headline is fake"""
    answer: int

In [19]:
def predict_headline(client, model, prompt):

    response = client.responses.parse(
        model=model, 
        instructions=None,
        input=prompt,
        text_format=FakeHeadlineOutput
    )

    return response.output_parsed

In [20]:
result = predict_headline(openai, "gpt-5-nano", test_set_frontier_LLM_ready[0]["prompt"])

In [21]:
result.answer

2